# LSTM / RNN OVERALL TRAINING NOTEBOOK

F1Tenth, Team 4 Final Project.

### Overview

This notebook is meant to be a baseline start for training and testing LSTM and RNN models with reference to MCP baselines.

It is split into relevant sections, and is meant to be highly modular allowing for changes to training data, model types, and methods.

## Section 0: Setup

Sets general parameters for the model, and sets the prediction length and batch size!

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import os
import numpy as np
import pandas as pd
import random
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

In [ ]:
# Feature selection
INPUT_FEATURES = ['velocity', 'steering_angle', 'lidar_scans']
TARGET_FEATURES = ['steering_angle', 'velocity']

# Hyperparameters
SEQ_LENGTH = 10       # How many past timesteps the model sees
PREDICT_LENGTH = 1    # How many timesteps into the future to predict
BATCH_SIZE = 64

# Device setup for GPU acceleration
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## Section 1: Dataset Loading and Cleaning

To faciliate proper training the datasets must be useful and also cleaned to ensure it is applicable to the training use case.

Main points.

1. Load data from either existing dataset or different MCP runs as a tensor.
Each row should be a type of data. Each column is a timestep

Predicted columns
a. velocity
b. steering angle
c. lidar scans (large array of inputs)
d. Additional data, depending on MCP, including particle filter location or track percentage stuff.

2. Mask data so that at each training step, the model can only see past and present information. This ensures realistic, useful training

3. Organize data into a Train, Val, Test split with a certain number of unique tracks in each for testing.

4. OPTIONAL (Save this data as a zip or even on a huggingface dataset for easy access"



In [ ]:
## TODO: Fill in function once general dataset is understood, normalize data!
# When running, this normalization will need to be integrated into the control policy
class F1TenthDataset(Dataset):
    """
    Custom PyTorch Dataset for rolling time-series windows.
    """
    def __init__(self, data_frame, input_cols, target_cols, seq_length, predict_length):
        self.seq_length = seq_length
        self.predict_length = predict_length

        # Extract the raw numpy arrays from the dataframe
        self.inputs = data_frame[input_cols].values
        self.targets = data_frame[target_cols].values

    def __len__(self):
        # Total valid sequences we can extract
        return len(self.inputs) - self.seq_length - self.predict_length + 1

    def __getitem__(self, idx):
        # Grab a window of 'seq_length' for the inputs
        x = self.inputs[idx : idx + self.seq_length]

        # Grab the future target(s)
        # If predict_length is 1, it predicts the very next timestep
        target_idx = idx + self.seq_length + self.predict_length - 1
        y = self.targets[target_idx]

        # Convert to PyTorch tensors
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

#TODO: Update as needed to fit the actual form of data in dataset(s)
def load_and_prep_data(df, input_cols, target_cols, seq_length, predict_length, batch_size):
    """
    Splits data by whole tracks, normalizes, and creates DataLoaders.
    Assumes your dataframe has a 'track_id' column.
    """
    # Get unique tracks to ensure we split by full tracks
    unique_tracks = df['track_id'].unique()

    # Define track split
    train_track_ids = unique_tracks[]
    val_track_ids = unique_tracks[]
    test_track_ids = unique_tracks[]

    # Separate the DataFrames by track ID
    train_df = df[df['track_id'].isin(train_track_ids)].copy()
    val_df = df[df['track_id'].isin(val_track_ids)].copy()
    test_df = df[df['track_id'].isin(test_track_ids)].copy()

    # Initialize the scaler
    scaler = MinMaxScaler()

    # Fit and transform training inputs to prevent data leakage
    train_df[input_cols] = scaler.fit_transform(train_df[input_cols])

    # Transform validation and test inputs using the training fit
    val_df[input_cols] = scaler.transform(val_df[input_cols])
    test_df[input_cols] = scaler.transform(test_df[input_cols])

    # Helper function to safely build sequences without crossing tracks
    def build_track_datasets(dataframe, track_ids):
        datasets = []
        for t_id in track_ids:
            track_data = dataframe[dataframe['track_id'] == t_id]

            # Skip tracks that are too short for the sequence window
            if len(track_data) > (seq_length + predict_length):
                ds = F1TenthDataset(track_data, input_cols, target_cols, seq_length, predict_length)
                datasets.append(ds)

        # ConcatDataset treats a list of datasets as one continuous dataset
        return ConcatDataset(datasets) if datasets else None

    # Build the combined datasets
    train_dataset = build_track_datasets(train_df, train_track_ids)
    val_dataset = build_track_datasets(val_df, val_track_ids)
    test_dataset = build_track_datasets(test_df, test_track_ids)

    # Create DataLoaders
    # Shuffle training so the model sees different track parts randomly
    # Keep validation and test unshuffled to visualize them sequentially later
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader, scaler

In [ ]:
# Load dataset
# df = pd.read_csv("all_mcp_runs.csv")

# Run data pipeline
# train_loader, val_loader, test_loader, fitted_scaler = load_and_prep_data(
#     df=df,
#     input_cols=INPUT_FEATURES,
#     target_cols=TARGET_FEATURES,
#     seq_length=SEQ_LENGTH,
#     predict_length=PREDICT_LENGTH,
#     batch_size=BATCH_SIZE
# )

## Section 2: Model Setup

The main focus of this project is LSTMs. To train these effectively there have been a number of different appraoches to improve LSTM training. Also, since we are using Pytorch, we can easily test other models like RNNs or GRUs if time allows

Main points to Consider

1. Type of model: RNN, LSTM, GRU

2. Number of model blocks: Generally between 1 and 3 in past studies

3. Size of hidden state size for model

4. Embedding layer before input? Can reduce input dimensionilty and "weight" each input to allow the LSTM to make better choices

5. Size and number of FC layers after LSTM

6. Option to load existing model from huggingface or .pkl file

Good hyperparameter options/ideas

https://ieeexplore.ieee.org/stamp/stamp.jsp?tp=&arnumber=10275226




In [ ]:

class F1TenthSequenceModel(nn.Module):
    def __init__(
        self,
        input_dim,
        rnn_hidden_dim,
        output_dim,
        model_type='LSTM',
        num_rnn_layers=2,
        use_embedding=True,
        embedding_dim=500,
        fc_layer_sizes=[256, 128, 32],
        dropout=0.2
    ):
        super(F1TenthSequenceModel, self).__init__()

        self.model_type = model_type.upper()
        self.use_embedding = use_embedding

        # Configure the Embedding Block (Dense / MLP style as per the paper)
        if self.use_embedding:
            self.embedding = nn.Sequential(
                nn.Linear(input_dim, embedding_dim),
                nn.ReLU()
            )
            rnn_input_dim = embedding_dim
        else:
            self.embedding = None
            rnn_input_dim = input_dim

        # Configure the Recurrent Core
        if self.model_type == 'LSTM':
            self.rnn = nn.LSTM(input_size=rnn_input_dim, hidden_size=rnn_hidden_dim, num_layers=num_rnn_layers, batch_first=True, dropout=dropout if num_rnn_layers > 1 else 0)
        elif self.model_type == 'GRU':
            self.rnn = nn.GRU(input_size=rnn_input_dim, hidden_size=rnn_hidden_dim, num_layers=num_rnn_layers, batch_first=True, dropout=dropout if num_rnn_layers > 1 else 0)
        else:
            self.rnn = nn.RNN(input_size=rnn_input_dim, hidden_size=rnn_hidden_dim, num_layers=num_rnn_layers, batch_first=True, dropout=dropout if num_rnn_layers > 1 else 0)

        # Dynamically build the Fully Connected output block
        fc_modules = []
        current_input_size = rnn_hidden_dim

        for hidden_size in fc_layer_sizes:
            fc_modules.append(nn.Linear(current_input_size, hidden_size))
            fc_modules.append(nn.ReLU())
            current_input_size = hidden_size

        # Add the final output layer (the paper uses tanh for the final output)
        fc_modules.append(nn.Linear(current_input_size, output_dim))
        fc_modules.append(nn.Tanh())

        self.fc = nn.Sequential(*fc_modules)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()

        # Pass through embedding if enabled
        if self.use_embedding:
            # Reshape to process all sequence steps through the linear layer efficiently
            x_reshaped = x.view(batch_size * seq_len, -1)
            embedded = self.embedding(x_reshaped)
            # Reshape back to sequence format for the RNN
            rnn_in = embedded.view(batch_size, seq_len, -1)
        else:
            rnn_in = x

        # Process through the selected RNN type
        if self.model_type == 'LSTM':
            rnn_out, (hidden, cell) = self.rnn(rnn_in)
        else:
            rnn_out, hidden = self.rnn(rnn_in)

        # Isolate the final timestep for prediction
        final_timestep_out = rnn_out[:, -1, :]
        predictions = self.fc(final_timestep_out)

        return predictions

def load_existing_model(model, filepath, device):
    """Loads weights from a .pkl or .pth file if it exists."""
    if os.path.exists(filepath):
        model.load_state_dict(torch.load(filepath, map_location=device))
        print(f"Successfully loaded model weights from {filepath}")
    else:
        print("File not found. Starting with randomly initialized weights.")
    return model



In [ ]:
#TODO: Need to check input dimensions!!
#the large number of inputs could cause our param count to increase a ton!
#Using the embedder (small MLP) could reduce dimensionality significantly


# model = F1TenthSequenceModel(
#     input_dim=1083,            # 1081 LiDAR + 1 Speed + 1 Steering
#     rnn_hidden_dim=1500,
#     output_dim=2,              # Throttle and Steering
#     model_type='LSTM',
#     num_rnn_layers=1,          # Paper uses SimpleRNN
#     use_embedding=True,
#     embedding_dim=500,
#     fc_layer_sizes=[256, 128, 32]
# )

## Section 3: Training and Hyperparameter Setup

Need to determine the best parameters to train with

Main points

1. Type of training: Teacher forcing vs Auto-regression. Can start with teacher forcing and fine-tune further w/ auto-regression?

2. Loss fuciton: basically just RMSE

3. Optimiser: likely ADAM

4. Activation Function: Relu or Leaky-Relu (GeLU?)

5. learning rate 1e-5, 1e-6, lr scheduler?

6. Gradient clipping, needed to avoid exploding grads

7. Dropout and other regularization

8. #of epochs

9. Extra? Early stopping and model saving!

In [ ]:

class RMSELoss(nn.Module):
    """
    Custom Root Mean Squared Error Loss.
    PyTorch does not have a built-in RMSE, so we wrap MSELoss.
    """
    def __init__(self):
        super(RMSELoss, self).__init__()
        self.mse = nn.MSELoss()

    def forward(self, predictions, targets):
        return torch.sqrt(self.mse(predictions, targets))

class EarlyStopping:
    """
    Halts training if validation loss stops improving after a set number of epochs.
    """
    def __init__(self, patience=7, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

# Training Hyperparameters
EPOCHS = 100
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5         # L2 Regularization to prevent overfitting
MAX_GRAD_NORM = 1.0         # Gradient clipping threshold

# Teacher Forcing configuration for multi-step predictions
# A ratio of 1.0 means full teacher forcing (always using ground truth for the next step)
# A ratio of 0.0 means full auto-regression (using the model's own predictions)
TEACHER_FORCING_RATIO = 1.0

# Early stopping and Scheduler configurations
EARLY_STOPPING_PATIENCE = 10
SCHEDULER_PATIENCE = 5
SCHEDULER_FACTOR = 0.5

# Initialize the loss function
criterion = RMSELoss()



In [ ]:
# We will initialize the optimizer and scheduler inside the training function
# or right before it, once the model is instantiated. Here is how they will look:

# optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(
#     optimizer,
#     mode='min',
#     factor=SCHEDULER_FACTOR,
#     patience=SCHEDULER_PATIENCE,
#     verbose=True
# )
# early_stopper = EarlyStopping(patience=EARLY_STOPPING_PATIENCE)

## Section 4: Training, Testing, Validation

Run training and print out RMSE loss for train, val, and test for each epoch.

Make a function to graph at each timestep

In [ ]:
def evaluate_model(model, dataloader, criterion, device):
    """
    Evaluates the model over a given dataloader and returns the average RMSE loss.
    """
    model.eval()
    batch_losses = []

    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            predictions = model(batch_x)
            loss = criterion(predictions, batch_y)
            batch_losses.append(loss.item())

    return np.mean(batch_losses)

def train_pipeline(
    model, train_loader, val_loader, test_loader,
    criterion, optimizer, scheduler, early_stopper,
    epochs, max_grad_norm, device
):
    """
    Main training loop tracking train, val, and test metrics.
    """
    history = {'train_rmse': [], 'val_rmse': [], 'test_rmse': []}

    for epoch in range(epochs):
        model.train()
        train_batch_losses = []

        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            optimizer.zero_grad()

            # Forward pass
            predictions = model(batch_x)
            loss = criterion(predictions, batch_y)

            # Backward pass
            loss.backward()

            # Gradient clipping to prevent exploding gradients in RNNs/LSTMs
            nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

            # Update weights
            optimizer.step()

            train_batch_losses.append(loss.item())

        # Calculate end-of-epoch metrics
        avg_train_loss = np.mean(train_batch_losses)
        avg_val_loss = evaluate_model(model, val_loader, criterion, device)
        avg_test_loss = evaluate_model(model, test_loader, criterion, device)

        # Store metrics for analytics
        history['train_rmse'].append(avg_train_loss)
        history['val_rmse'].append(avg_val_loss)
        history['test_rmse'].append(avg_test_loss)

        # Update learning rate and check for early stopping
        scheduler.step(avg_val_loss)
        early_stopper(avg_val_loss)

        print(f"Epoch [{epoch+1:03d}/{epochs:03d}] | Train RMSE: {avg_train_loss:.4f} | Val RMSE: {avg_val_loss:.4f} | Test RMSE: {avg_test_loss:.4f}")

        if early_stopper.early_stop:
            print("Early stopping triggered. Halting training to prevent overfitting.")
            break

    return history


In [ ]:
# Execution:
# history = train_pipeline(
#     model=model,
#     train_loader=train_loader,
#     val_loader=val_loader,
#     test_loader=test_loader,
#     criterion=criterion,
#     optimizer=optimizer,
#     scheduler=scheduler,
#     early_stopper=early_stopper,
#     epochs=EPOCHS,
#     max_grad_norm=MAX_GRAD_NORM,
#     device=DEVICE
# )

## Section 5: Analytics

Output full graphs of model losses in all cases

Additional analytics?

In [ ]:


def plot_learning_curves(history):
    """
    Plots the RMSE loss for train, validation, and test sets over all epochs.
    """
    plt.figure(figsize=(10, 6))
    plt.plot(history['train_rmse'], label='Train RMSE', color='blue', linewidth=2)
    plt.plot(history['val_rmse'], label='Validation RMSE', color='orange', linewidth=2)
    plt.plot(history['test_rmse'], label='Test RMSE', color='green', linewidth=2, linestyle='--')

    plt.title('Model RMSE Loss Over Epochs', fontsize=14)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('RMSE', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()

def plot_predictions_vs_actual(model, dataloader, device, num_samples=150):
    """
    Graphs the model's predictions against actual telemetry at each timestep.
    Grabs a continuous chunk of data from the provided dataloader.
    """
    model.eval()
    all_preds = []
    all_actuals = []

    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            batch_x = batch_x.to(device)
            preds = model(batch_x)

            all_preds.extend(preds.cpu().numpy())
            all_actuals.extend(batch_y.numpy())

            if len(all_preds) >= num_samples:
                break

    # Truncate to the requested number of samples
    all_preds = np.array(all_preds)[:num_samples]
    all_actuals = np.array(all_actuals)[:num_samples]

    # Create subplots for each target feature
    # Assumes TARGET_FEATURES = ['steering_angle', 'velocity']
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    # Plot Steering Angle
    axes[0].plot(all_actuals[:, 0], label='Actual Steering', color='black')
    axes[0].plot(all_preds[:, 0], label='Predicted Steering', color='red', linestyle='--')
    axes[0].set_title('Steering Angle: Actual vs Predicted')
    axes[0].set_ylabel('Normalized Angle')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Plot Velocity
    axes[1].plot(all_actuals[:, 1], label='Actual Velocity', color='black')
    axes[1].plot(all_preds[:, 1], label='Predicted Velocity', color='blue', linestyle='--')
    axes[1].set_title('Velocity: Actual vs Predicted')
    axes[1].set_ylabel('Normalized Velocity')
    axes[1].set_xlabel('Timestep')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()



In [ ]:
# Example Execution:
# plot_learning_curves(history)
# plot_predictions_vs_actual(model, test_loader, DEVICE, num_samples=200)

## Section 6: Model Saving

Save model as a .pkl file or to hugging face or something if good!

Could also allow for future tuning and training

In [ ]:
import joblib

def save_model_and_scaler(model, scaler, model_filename="f1tenth_model.pth", scaler_filename="f1tenth_scaler.pkl"):
    """
    Saves the PyTorch model weights and the scikit-learn data scaler.
    Both are required for deployment to ensure live sensor data is normalized correctly.
    """
    # Save the PyTorch model architecture weights
    model_save_path = os.path.join(os.getcwd(), model_filename)
    torch.save(model.state_dict(), model_save_path)
    print(f"Model weights successfully saved to: {model_save_path}")

    # Save the scikit-learn scaler object
    scaler_save_path = os.path.join(os.getcwd(), scaler_filename)
    joblib.dump(scaler, scaler_save_path)
    print(f"Data scaler successfully saved to: {scaler_save_path}")



In [ ]:
# Example Execution:
# save_model_and_scaler(
#     model=model,
#     scaler=fitted_scaler,
#     model_filename=f"f1tenth_{MODEL_TYPE.lower()}_best.pth",
#     scaler_filename="f1tenth_data_scaler.pkl"
# )

# ==========================================
# Example: Loading for Deployment / Fine-Tuning
# ==========================================
# When deploying to the car, you would recreate the model architecture
# and load the weights and scaler like this:
#NOTE! when creating actual testin scripts for sim or real, need to ensure
#input to control policy are normalized identically!

# loaded_model = F1TenthSequenceModel(
#     input_dim=1083,
#     rnn_hidden_dim=1500,
#     output_dim=2,
#     model_type='LSTM',
#     use_embedding=True
# ).to(DEVICE)
#
# loaded_model.load_state_dict(torch.load("f1tenth_lstm_best.pth", map_location=DEVICE))
# loaded_model.eval()
#
# loaded_scaler = joblib.load("f1tenth_data_scaler.pkl")